## Cleaning Pipeline Introduction

The cleaning pipeline is used to systematically identify, validate, and isolate problematic records in the main dataset before rebalancing. It applies a consistent set of quality checks to the audio files and records the results in a manifest, allowing files that do not meet the required dataset criteria to be identified.

The pipeline focuses on ensuring that the dataset contains readable and valid audio files with consistent formats and acceptable audio characteristics. Issues such as unreadable files, invalid formats, unsuitable durations, excessive silence, and other quality problems can be detected through these checks.

Rather than manually inspecting files, the pipeline provides an automated and reproducible cleaning process. Each file is assessed using the same validation criteria, and the results are recorded so that problematic records can be reviewed and removed or handled appropriately.

This is important for Sprint 2 because the quality of the dataset must be established before rebalancing. Rebalancing an unclean dataset could result in poor-quality or invalid records being carried into the final balanced dataset. The cleaning pipeline therefore provides the foundation for producing a clean, validated, and reliable dataset that can subsequently be balanced and prepared for augmentation.

In [1]:
from pathlib import Path
import zipfile
import shutil
import hashlib
import re
import os
import pandas as pd

## Dataset Extraction and Initial Inventory

This step locates the dataset ZIP file in the current Jupyter working directory, checks that a ZIP file is available, records the selected file and its size, and then extracts the dataset into an `original_dataset` folder. After extraction, all files are identified recursively and the total number of files and overall extracted dataset size are calculated.

This is important to the Sprint 2 cleaning pipeline because it establishes a **clear and reproducible starting point for dataset validation**. Before cleaning can begin, the pipeline needs to know which dataset is being processed and confirm that the expected data has been successfully extracted. Recording the initial file count and dataset size also provides a baseline that can later be compared against the cleaned and balanced dataset. This allows changes made during cleaning and rebalancing to be tracked and verified, supporting the Sprint 2 requirements for a **cleaned, validated, reproducible, and augmentation-ready dataset**.

In [4]:
#get the file 


CURRENT_DIR = Path.cwd()

print("Jupyter folder:")
print(CURRENT_DIR)

print("\nZIP files found:")

for p in CURRENT_DIR.glob("*.zip"):
    print(p.name)
zip_files = list(CURRENT_DIR.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError("No ZIP file found in the Jupyter folder.")

if len(zip_files) > 1:
    print("Multiple ZIP files found:")
    for i, p in enumerate(zip_files):
        print(i, p.name)
else:
    print("Using:", zip_files[0].name)
ZIP_PATH = zip_files[0]

print("ZIP:", ZIP_PATH)
print(
    "ZIP size:",
    round(ZIP_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

Jupyter folder:
C:\Users\ASUS\Documents

ZIP files found:
dataset.zip
Using: dataset.zip
ZIP: C:\Users\ASUS\Documents\dataset.zip
ZIP size: 772.19 MB


In [5]:
#Extract 
EXTRACT_ROOT = CURRENT_DIR / "original_dataset"

EXTRACT_ROOT.mkdir(exist_ok=True)

print("Extracting...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_ROOT)

print("Extraction complete.")
print("Extracted to:", EXTRACT_ROOT)

all_files = [
    p for p in EXTRACT_ROOT.rglob("*")
    if p.is_file()
]

print("Total files:", len(all_files))

total_size_gb = sum(
    p.stat().st_size for p in all_files
) / (1024 ** 3)

print(f"Total extracted size: {total_size_gb:.2f} GB")

Extracting...
Extraction complete.
Extracted to: C:\Users\ASUS\Documents\original_dataset
Total files: 15574
Total extracted size: 0.80 GB


## Load Dataset QA Reports

This step defines the project root and the location of the validation reports generated by the Data Quality Assurance (QA) pipeline. It then specifies the paths to the dataset manifest, file-duplicate report, and metadata-duplicate report.

The `dataset_manifest.csv` contains the validation results for the dataset records, while the duplicate reports identify potential duplicate files and duplicate records based on their metadata.

This is important to the Sprint 2 cleaning pipeline because it **reuses the existing Data QA validation results rather than repeating the initial quality checks from scratch**. These reports provide the information required to identify unreadable, invalid, duplicated, or inconsistent records and determine which records need to be addressed during cleaning. Using the generated reports also creates a traceable link between the earlier QA process and the Sprint 2 cleaning and validation workflow, supporting consistency and reproducibility.

In [29]:
PROJECT_ROOT = Path.cwd()

VALIDATION_DIR = PROJECT_ROOT / "validation_reports"

MANIFEST_PATH = VALIDATION_DIR / "dataset_manifest.csv"
FILE_DUPLICATES_PATH = VALIDATION_DIR / "file_duplicates.csv"
METADATA_DUPLICATES_PATH = VALIDATION_DIR / "metadata_duplicates.csv"

print("Project root:", PROJECT_ROOT)
print("Validation directory:", VALIDATION_DIR)

Project root: C:\Users\ASUS\Documents
Validation directory: C:\Users\ASUS\Documents\validation_reports


In [30]:
for path in [
    MANIFEST_PATH,
    FILE_DUPLICATES_PATH,
    METADATA_DUPLICATES_PATH
]:
    print(
        "✓" if path.exists() else "✗",
        path
    )

✓ C:\Users\ASUS\Documents\validation_reports\dataset_manifest.csv
✓ C:\Users\ASUS\Documents\validation_reports\file_duplicates.csv
✓ C:\Users\ASUS\Documents\validation_reports\metadata_duplicates.csv


In [31]:
manifest = pd.read_csv(MANIFEST_PATH)

print("Validation manifest loaded.")
print("Records:", len(manifest))

print("\nColumns:")
print(manifest.columns.tolist())

Validation manifest loaded.
Records: 15574

Columns:
['file_name', 'absolute_path', 'relative_path', 'extension', 'size_bytes', 'source', 'species', 'species_label', 'format_status', 'readable', 'duration', 'sample_rate', 'channels', 'issue']


In [32]:
file_duplicates = pd.read_csv(
    FILE_DUPLICATES_PATH
)

print(
    "File duplicate records:",
    len(file_duplicates)
)

print(
    file_duplicates.columns.tolist()
)

metadata_duplicates = pd.read_csv(
    METADATA_DUPLICATES_PATH
)

print(
    "Metadata duplicate records:",
    len(metadata_duplicates)
)

File duplicate records: 10
['file_name', 'absolute_path', 'relative_path', 'extension', 'size_bytes', 'source', 'species', 'species_label', 'sha256']
Metadata duplicate records: 0


In [33]:
clean_manifest = manifest.copy()

clean_manifest["cleaning_action"] = "keep"
clean_manifest["cleaning_reason"] = ""

## 1. Unsupported Audio Formats

This step defines the audio file formats that are supported by the cleaning pipeline and then searches through the extracted dataset to keep only files with those supported extensions. Files that are not audio files or use an unsupported format are excluded from the list of files that will continue through the cleaning process.

The file extension is converted to lowercase before checking, so formats such as `.WAV` and `.wav` are treated consistently.

This is important to the Sprint 2 cleaning pipeline because **unsupported or inconsistent file formats should not proceed into the validation and rebalancing stages**. Restricting the dataset to supported audio formats ensures that subsequent audio-quality checks can be applied consistently and reduces the risk of incompatible files being included in the cleaned dataset. The resulting count also provides a baseline for tracking how many supported audio files are available after this initial cleaning step.


In [6]:
#start cleaning - 
SUPPORTED_AUDIO_EXTENSIONS = {
    ".wav",
    ".mp3",
    ".flac",
    ".ogg"
}

audio_files = [
    p for p in EXTRACT_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in SUPPORTED_AUDIO_EXTENSIONS
]

print("Audio files found:", len(audio_files))

Audio files found: 15552


## Cleaning Invalid, Inconsistent and Duplicate Records

This stage applies the main cleaning rules to the dataset using the validation results stored in the cleaning manifest. Each cleaning rule identifies a specific type of problematic record and records the required action and reason in the manifest.

The records are **marked for removal rather than immediately deleted**. This keeps a traceable record of what was identified during cleaning and why it was removed, supporting reproducibility and allowing the cleaning results to be reviewed.

### Step 1 — Remove Empty Files

Files with a size of zero bytes, or with missing size information treated as zero, are identified and marked for removal with the reason `empty_file`.

Empty files do not contain usable audio data and therefore should not proceed to the later validation, balancing, or augmentation stages.

**Why this is important:**  
Removing empty files ensures that the cleaned dataset only contains records with actual data and prevents unusable files from affecting later stages of the pipeline.

### Step 2 — Remove Unreadable Files

Files that are not identified as readable by the validation results are marked for removal with the reason `unreadable`.

These files cannot be reliably processed or used as audio samples.

**Why this is important:**  
Unreadable records are unusable for downstream processing and could cause errors during later audio analysis, balancing, or augmentation. Removing them ensures that the dataset contains files that can be successfully processed.

### Step 3 — Remove Files Shorter Than One Second

Audio files with a duration below `1.0` second are identified and marked for removal with the reason `too_short_less_than_1_second`.

Very short recordings may not contain enough useful audio information and would introduce inconsistent sample lengths into the dataset.

**Why this is important:**  
Applying a consistent minimum-duration requirement improves the quality and consistency of the dataset before rebalancing and helps ensure that unsuitable recordings are not passed to the augmentation stage.

### Step 4 — Standardise Species Names

The original species labels are first preserved in `species_label_original` so that any changes can be tracked.

The standardisation process then:
- Converts missing labels to `unknown`
- Removes leading and trailing spaces
- Converts underscores into spaces
- Converts multiple spaces into a single space

The `species_changes` table records the original and standardised labels that were changed.

**Why this is important:**  
Consistent species labels are essential for correctly grouping records into classes. Without standardisation, the same species could appear under slightly different labels and be incorrectly treated as separate classes during rebalancing.

### Step 5 — Remove Exact Duplicates Across Buckets

The duplicate reports are used to identify exact duplicate audio files between `bucket_1` and `bucket_3`.

The pipeline:
1. Identifies files belonging to `bucket_1` and `bucket_3`.
2. Collects the SHA-256 hashes of files in `bucket_1`.
3. Finds files in `bucket_3` with matching SHA-256 hashes.
4. Keeps the `bucket_1` copy.
5. Marks the duplicate `bucket_3` copy for removal with the reason `exact_duplicate_keep_bucket1`.

**Why this is important:**  
Duplicate recordings can artificially increase the number of samples and distort the class distribution. They can also introduce data leakage if the same recording appears more than once. Removing exact duplicates ensures that each recording is represented only once in the cleaned dataset.

### Step 6 — Remove Unsupported Audio Formats

The cleaning manifest is checked against the supported audio extensions:

- `.wav`
- `.mp3`
- `.flac`
- `.ogg`

Any record with an unsupported extension is marked for removal with the reason `unsupported_extension`.

**Why this is important:**  
Restricting the dataset to supported formats ensures that subsequent audio-processing and validation steps can be applied consistently and reduces the risk of incompatible files entering the cleaned dataset.

### Overall Cleaning Outcome

These steps establish the main quality-control stage before rebalancing. The pipeline identifies and marks records that are:

- Empty
- Unreadable
- Too short
- Inconsistently labelled
- Exact duplicates
- Stored in unsupported formats

By recording a `cleaning_action` and `cleaning_reason` for each affected record, the process remains **traceable and reproducible**.

This is important to Sprint 2 because it ensures that **rebalancing is performed on a clean and consistent dataset**, rather than balancing data that still contains invalid or problematic records. The resulting cleaned dataset therefore provides a reliable foundation for the final balanced-dataset validation and subsequent augmentation stage.


In [34]:
#remove empty
empty_mask = (
    clean_manifest["size_bytes"].fillna(0) == 0
)

clean_manifest.loc[
    empty_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    empty_mask,
    "cleaning_reason"
] = "empty_file"

In [35]:
#remove unreadable
unreadable_mask = (
    clean_manifest["readable"] != True
)

clean_manifest.loc[
    unreadable_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    unreadable_mask,
    "cleaning_reason"
] = "unreadable"

In [36]:
#remove files under 1 second 
too_short_mask = (
    clean_manifest["duration"].notna()
    & (clean_manifest["duration"] < 1.0)
)

clean_manifest.loc[
    too_short_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    too_short_mask,
    "cleaning_reason"
] = "too_short_less_than_1_second"

In [37]:
#standadrise species names 
clean_manifest["species_label_original"] = (
    clean_manifest["species_label"]
)
def standardise_species_name(name):

    if pd.isna(name):
        return "unknown"

    name = str(name).strip()

    # Underscores → spaces
    name = name.replace("_", " ")

    # Multiple spaces → one space
    name = re.sub(r"\s+", " ", name)

    return name

clean_manifest["species_label"] = (
    clean_manifest["species_label"]
    .apply(standardise_species_name)
)

In [38]:
species_changes = (
    clean_manifest[
        clean_manifest["species_label_original"]
        != clean_manifest["species_label"]
    ][
        [
            "species_label_original",
            "species_label"
        ]
    ]
    .drop_duplicates()
    .sort_values("species_label")
)

species_changes

,species_label_original,species_label
12514,acanthiza_chrysorrhoa,acanthiza chrysorrhoa
14437,acanthiza_lineata,acanthiza lineata
983,acanthiza_nana,acanthiza nana
13759,acanthiza_pusilla,acanthiza pusilla
10191,acanthiza_reguloides,acanthiza reguloides
...,...,...
11133,trichosurus_vulpecula,trichosurus vulpecula
0,uperoleia_altissima,uperoleia altissima
134,uperoleia_mimula,uperoleia mimula
12612,vanellus_miles,vanellus miles


In [39]:
#duplicates keep the bucket 1 and remove bucket 3 
file_duplicates["is_bucket_1"] = (
    file_duplicates["relative_path"]
    .str.contains(
        "bucket_1",
        case=False,
        na=False
    )
)

file_duplicates["is_bucket_3"] = (
    file_duplicates["relative_path"]
    .str.contains(
        "bucket_3",
        case=False,
        na=False
    )
)

bucket1_hashes = set(
    file_duplicates.loc[
        file_duplicates["is_bucket_1"],
        "sha256"
    ].dropna()
)

bucket3_to_remove = file_duplicates[
    file_duplicates["is_bucket_3"]
    & file_duplicates["sha256"].isin(
        bucket1_hashes
    )
].copy()

print(
    "Bucket 3 exact duplicates to remove:",
    len(bucket3_to_remove)
)

Bucket 3 exact duplicates to remove: 5


In [40]:
bucket3_paths = set(
    bucket3_to_remove["relative_path"]
)
bucket3_mask = (
    clean_manifest["relative_path"]
    .isin(bucket3_paths)
)
clean_manifest.loc[
    bucket3_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    bucket3_mask,
    "cleaning_reason"
] = "exact_duplicate_keep_bucket1"

In [41]:
# Remove unsupported formats from the cleaning manifest

SUPPORTED_AUDIO_EXTENSIONS = {
    ".wav",
    ".mp3",
    ".flac",
    ".ogg"
}

unsupported_mask = ~(
    clean_manifest["extension"]
    .str.lower()
    .isin(SUPPORTED_AUDIO_EXTENSIONS)
)

clean_manifest.loc[
    unsupported_mask,
    "cleaning_action"
] = "remove"

clean_manifest.loc[
    unsupported_mask,
    "cleaning_reason"
] = "unsupported_extension"

## Cleaning Removal Summary

This step provides an overall summary of the cleaning decisions made by the pipeline. It uses the `clean_manifest` to identify all records marked for removal and groups them by the reason they were removed.

The `removal_summary` first filters the manifest to records where `cleaning_action` is `"remove"`. It then groups these records by `cleaning_reason` and counts the number of files in each category. Sorting the results from highest to lowest count makes it easier to identify which types of data-quality problems were most common.

The code then reports three key dataset counts:

- **Original files** — the total number of records present in the cleaning manifest before removal.
- **Files to remove** — the number of records that have been identified as unsuitable and marked with `"remove"`.
- **Files to keep** — the number of records that have passed the cleaning checks and are marked with `"keep"`.

This is important to the Sprint 2 cleaning pipeline because it provides a **quantitative check of the cleaning stage**. The removal breakdown shows what types of problems were found, while the original, removed, and retained counts show how the dataset has changed as a result of cleaning. These counts provide a baseline for the subsequent rebalancing stage and help verify that the cleaning process has been applied consistently and that the dataset is ready to proceed to the next stage.


In [42]:
#chekc removal summary 
removal_summary = (
    clean_manifest[
        clean_manifest["cleaning_action"] == "remove"
    ]
    .groupby("cleaning_reason")
    .size()
    .reset_index(name="file_count")
    .sort_values(
        "file_count",
        ascending=False
    )
)

removal_summary

print(
    "Original files:",
    len(clean_manifest)
)

print(
    "Files to remove:",
    (
        clean_manifest["cleaning_action"]
        == "remove"
    ).sum()
)

print(
    "Files to keep:",
    (
        clean_manifest["cleaning_action"]
        == "keep"
    ).sum()
)

Original files: 15574
Files to remove: 747
Files to keep: 14827


## Cleaning Validation and Overlap Check

This step verifies that the cleaning rules have been applied consistently and checks whether records are being identified by more than one cleaning criterion.

The cleaning manifest contains **15,574 records** in total. The individual validation checks identified **722 files shorter than one second, 22 files with unsupported formats, 22 unreadable files, and 0 empty files**. After accounting for records that may satisfy more than one condition, **747 records are currently marked for removal**.

The individual categories do not simply add together because some files can appear in more than one category. Therefore, the pipeline performs an overlap check between the three relevant categories:

- **Short files:** 722
- **Unsupported files:** 22
- **Bucket 3 duplicate files:** 5
- **Short + unsupported:** 0
- **Short + duplicate:** 2
- **Unsupported + duplicate:** 0
- **All three:** 0

The overlap results show that **2 files are both shorter than one second and identified as Bucket 3 duplicates**. There is no overlap between the short and unsupported categories, no overlap between unsupported and duplicate files, and no file belongs to all three categories.

This explains why the number of files marked for removal (**747**) is lower than simply adding all individual categories together. The cleaning rules operate together, so a file that meets multiple removal criteria is only counted once in the final removal total.

This check is important to Sprint 2 because it provides a **consistency check on the cleaning process** and confirms that the removal count is being interpreted correctly. It also shows that the cleaning rules are not unnecessarily counting the same record multiple times. Establishing these counts before rebalancing provides a reliable baseline for the cleaned dataset and supports the requirement for a **validated and reproducible dataset**.


In [43]:
print("Manifest total:", len(clean_manifest))

print("\nExpected individual categories:")
print(
    "Under 1 sec:",
    (
        clean_manifest["duration"].notna()
        & (clean_manifest["duration"] < 1)
    ).sum()
)

print(
    "Unsupported:",
    (
        ~clean_manifest["extension"]
        .str.lower()
        .isin(SUPPORTED_AUDIO_EXTENSIONS)
    ).sum()
)

print(
    "Unreadable:",
    (clean_manifest["readable"] != True).sum()
)

print(
    "Empty:",
    (clean_manifest["size_bytes"].fillna(0) == 0).sum()
)

print(
    "Currently marked remove:",
    (
        clean_manifest["cleaning_action"]
        == "remove"
    ).sum()
)

Manifest total: 15574

Expected individual categories:
Under 1 sec: 722
Unsupported: 22
Unreadable: 22
Empty: 0
Currently marked remove: 747


In [44]:
short_mask = (
    clean_manifest["duration"].notna()
    & (clean_manifest["duration"] < 1)
)

unsupported_mask = ~(
    clean_manifest["extension"]
    .str.lower()
    .isin(SUPPORTED_AUDIO_EXTENSIONS)
)

duplicate_mask = (
    clean_manifest["relative_path"]
    .isin(bucket3_paths)
)

print("Short:", short_mask.sum())
print("Unsupported:", unsupported_mask.sum())
print("Bucket 3 duplicates:", duplicate_mask.sum())

print("\nShort + unsupported:",
      (short_mask & unsupported_mask).sum())

print("Short + duplicate:",
      (short_mask & duplicate_mask).sum())

print("Unsupported + duplicate:",
      (unsupported_mask & duplicate_mask).sum())

print("All three:",
      (short_mask & unsupported_mask & duplicate_mask).sum())

Short: 722
Unsupported: 22
Bucket 3 duplicates: 5

Short + unsupported: 0
Short + duplicate: 2
Unsupported + duplicate: 0
All three: 0


In [45]:
short_and_duplicate = clean_manifest[
    short_mask & duplicate_mask
].copy()

short_and_duplicate[
    [
        "file_name",
        "relative_path",
        "species_label",
        "duration",
        "extension",
        "size_bytes",
        "cleaning_action",
        "cleaning_reason"
    ]
]

,file_name,relative_path,species_label,duration,extension,size_bytes,cleaning_action,cleaning_reason
14389,project_echo_bucket_3__Ground_Parrot (14).wav,GCP/Pezoporus wallicus/project_echo_bucket_3__...,pezoporus wallicus,0.35,.wav,61784,remove,exact_duplicate_keep_bucket1
14425,project_echo_bucket_3__Ground_Parrot (11).wav,GCP/Pezoporus wallicus/project_echo_bucket_3__...,pezoporus wallicus,0.95,.wav,167624,remove,exact_duplicate_keep_bucket1


## Cleaning Verification and Species Distribution

This step performs a final verification of the cleaning decisions before the dataset moves further through the pipeline. The purpose is to confirm that records identified as problematic by the cleaning rules have not accidentally remained marked as `"keep"`.

The first check identifies files that are **shorter than one second but are still marked as `"keep"`**. The expected result is:

**Short files still being kept: 0**

This confirms that all files below the minimum duration threshold have been correctly identified for removal.

The second check identifies **unsupported audio formats that are still marked as `"keep"`**. The expected result is:

**Unsupported files still being kept: 0**

This confirms that unsupported formats have not accidentally remained in the retained dataset.

The third check identifies **Bucket 3 duplicate files that are still marked as `"keep"`**. The expected result is:

**Bucket-3 duplicates still being kept: 0**

This confirms that the identified duplicate records have been correctly marked for removal while the intended Bucket 1 copies are retained.

The final section calculates the **species distribution** of the `final_manifest` by grouping records according to `species_label` and counting the number of files in each species.

This is important to Sprint 2 because the verification checks provide evidence that the cleaning rules have actually been applied correctly rather than simply assuming that the earlier filtering worked. A result of zero for the three verification checks confirms that no identified short, unsupported, or duplicate records have accidentally been retained.

The species distribution is also important because it provides the **baseline class distribution after cleaning**. This information is needed before rebalancing so that the dataset can be assessed for class imbalance and the appropriate balancing strategy can be applied. It also allows the final balanced dataset to be compared against the cleaned dataset to verify that rebalancing has produced the intended class distribution.


In [46]:
##verfications 
short_kept = clean_manifest[
    (
        clean_manifest["duration"].notna()
        & (clean_manifest["duration"] < 1.0)
        & (clean_manifest["cleaning_action"] == "keep")
    )
]

print(
    "Short files still being kept:",
    len(short_kept)
)

unsupported_kept = clean_manifest[
    (
        ~clean_manifest["extension"]
        .str.lower()
        .isin(SUPPORTED_AUDIO_EXTENSIONS)
    )
    &
    (
        clean_manifest["cleaning_action"] == "keep"
    )
]

print(
    "Unsupported files still being kept:",
    len(unsupported_kept)
)

duplicate_kept = clean_manifest[
    duplicate_mask
    &
    (
        clean_manifest["cleaning_action"] == "keep"
    )
]

print(
    "Bucket-3 duplicates still being kept:",
    len(duplicate_kept)
)

Short files still being kept: 0
Unsupported files still being kept: 0
Bucket-3 duplicates still being kept: 0


In [47]:
#final clenaed manifest 
final_manifest = clean_manifest[
    clean_manifest["cleaning_action"] == "keep"
].copy()

print(
    "Final cleaned records:",
    len(final_manifest)
)

Final cleaned records: 14827


In [49]:
species_distribution = (
    final_manifest
    .groupby("species_label")
    .size()
    .reset_index(name="file_count")
)

species_distribution

,species_label,file_count
0,acanthiza chrysorrhoa,62
1,acanthiza lineata,58
2,acanthiza nana,258
3,acanthiza pusilla,480
4,acanthiza reguloides,268
...,...,...
117,trichosurus vulpecula,14
118,uperoleia altissima,130
119,uperoleia mimula,90
120,vanellus miles,108


In [53]:
# Look for the GCP folder in the current project
gcp_folders = list(
    PROJECT_ROOT.rglob("GCP")
)

for folder in gcp_folders:
    print(folder)

C:\Users\ASUS\Documents\original_dataset\dataset\GCP


In [54]:
ORIGINAL_DATASET_ROOT = (
    gcp_folders[0].parent
)

print(ORIGINAL_DATASET_ROOT)

missing_files = []

for _, row in final_manifest.iterrows():

    source = (
        ORIGINAL_DATASET_ROOT
        / row["relative_path"]
    )

    if not source.exists():
        missing_files.append(str(source))

print("Missing files:", len(missing_files))

print(
    "Original dataset exists:",
    ORIGINAL_DATASET_ROOT.exists()
)

C:\Users\ASUS\Documents\original_dataset\dataset
Missing files: 0
Original dataset exists: True


## Create and Verify the Cleaned Dataset

This step creates the physical `cleaned_dataset` using the records retained in `final_manifest`. The manifest acts as the source of truth, so only files that have passed the cleaning process are copied from the original dataset into the cleaned dataset.

The cleaned dataset is created as a separate directory from the original dataset. Each retained file is organised into a folder based on its standardised `species_label`, creating a consistent class-based structure for the next stage of the pipeline.

The pipeline then verifies that every file listed in `final_manifest` was successfully copied to the expected location. Any file that cannot be found is recorded in `missing_cleaned`, and the total number of missing files is reported.

The expected verification results are:

- **Missing cleaned files: 0** — confirms that every retained record from the final manifest has been successfully copied.
- **Cleaned dataset exists: True** — confirms that the cleaned dataset directory was successfully created.

This is important to Sprint 2 because it confirms that the cleaning decisions recorded in the manifest have been correctly converted into the **actual cleaned dataset**. It also ensures that no expected files are missing before the dataset moves into the rebalancing stage. Keeping the original and cleaned datasets separate supports **traceability and reproducibility**, while organising the files by standardised species labels provides a consistent structure for balancing and subsequent augmentation.


In [51]:
#create clened data set 
CLEANED_ROOT = (
    PROJECT_ROOT / "cleaned_dataset"
)

CLEANED_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Cleaned dataset:",
    CLEANED_ROOT
)

Cleaned dataset: C:\Users\ASUS\Documents\cleaned_dataset


In [55]:
for _, row in final_manifest.iterrows():

    source = (
        ORIGINAL_DATASET_ROOT
        / row["relative_path"]
    )

    species_folder = (
        CLEANED_ROOT
        / row["species_label"]
    )

    destination = (
        species_folder
        / row["file_name"]
    )

    species_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    shutil.copy2(
        source,
        destination
    )

In [58]:
#verify and rpeorts 
# Verify and report missing cleaned files

missing_cleaned = []

for _, row in final_manifest.iterrows():

    cleaned_path = (
        CLEANED_ROOT
        / row["species_label"]
        / row["file_name"]
    )

    if not cleaned_path.exists():
        missing_cleaned.append(str(cleaned_path))

print("Missing cleaned files:", len(missing_cleaned))

print(
    "Cleaned dataset exists:",
    CLEANED_ROOT.exists()
)

Missing cleaned files: 0
Cleaned dataset exists: True


## Save Cleaning Reports

This step creates a dedicated `cleaning_reports` directory and saves the results of the cleaning process into separate CSV files. These reports provide a permanent record of the cleaning decisions, removed records, removal reasons, and species distribution after cleaning.

The `cleaning_manifest.csv` contains the complete cleaning manifest, including the validation information and cleaning decisions recorded for each dataset record. The `final_cleaned_manifest.csv` contains the records that remain after the cleaning process and therefore represents the dataset used to create the cleaned dataset.

The `removed_files.csv` contains only the records marked with `cleaning_action == "remove"`. This provides a clear record of which files were excluded from the cleaned dataset and allows the reasons for their removal to be reviewed.

The `removal_summary.csv` stores the summary of removals by cleaning reason, showing how many files were removed for each identified issue. This provides a high-level overview of the impact of the cleaning process.

Finally, `species_distribution_after_cleaning.csv` records the number of files available for each species after cleaning. This is important for assessing the class distribution before rebalancing and determining which species may require adjustment.

This is important to Sprint 2 because these reports provide **evidence and documentation of the cleaning process**. They make the process traceable and reproducible by preserving what was removed, what was retained, why records were removed, and what the dataset distribution looked like after cleaning. These outputs also provide the information needed to move confidently from **cleaning and validation into rebalancing**.


In [59]:
CLEANING_REPORTS = (
    PROJECT_ROOT / "cleaning_reports"
)

CLEANING_REPORTS.mkdir(
    parents=True,
    exist_ok=True
)

clean_manifest.to_csv(
    CLEANING_REPORTS
    / "cleaning_manifest.csv",
    index=False
)

final_manifest.to_csv(
    CLEANING_REPORTS
    / "final_cleaned_manifest.csv",
    index=False
)

removed_files = clean_manifest[
    clean_manifest["cleaning_action"] == "remove"
].copy()

removed_files.to_csv(
    CLEANING_REPORTS
    / "removed_files.csv",
    index=False
)

removal_summary.to_csv(
    CLEANING_REPORTS
    / "removal_summary.csv",
    index=False
)

species_distribution.to_csv(
    CLEANING_REPORTS
    / "species_distribution_after_cleaning.csv"
)

In [ ]:
#Sprint 2
#Raveesha Somawansa
#s225559348